# SIRCH Phase 3 - P3-G - Entrainement RWF-2000 uniquement, test RLVS

Objectif : vraie generalisation cross-dataset sans melange des datasets.

Garanties integrees : checkpoint apres chaque epoch, EarlyStopping avec restore_best_weights=True, reprise automatique depuis le dernier checkpoint, CSVLogger separe.

In [ ]:
# ============================================================
# CELLULE 1 - Parametres SIRCH P3-G (rwf_only_on_rlvs)
# ============================================================
import csv
import glob
import os
import random
import re
import time
from pathlib import Path

import cv2
import numpy as np
import tensorflow as tf
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score
from sklearn.model_selection import train_test_split

EXPERIMENT_NAME = 'rwf_only_on_rlvs'
TRAIN_DOMAIN = 'RWF-2000'
TEST_DOMAIN = 'RLVS'

N_FRAMES = 20
IMG_SIZE = 224
LSTM_UNITS = 256
DROPOUT = 0.5

BATCH_SIZE = 8
EPOCHS = 30
LEARNING_RATE = 1e-4
OPTIMIZER = 'adam'
LOSS = 'binary_crossentropy'
VALIDATION_SIZE = 0.15
THRESHOLD = 0.5

DATASETS_ROOT = Path(os.environ.get('SIRCH_DATASETS_DIR', r'C:\SIRCH_ENV\datasets'))
PHASE3_MODEL_DIR = Path(os.environ.get('SIRCH_PHASE3_MODEL_DIR', r'C:\SIRCH_ENV\models\phase3'))
PHASE3_MODEL_DIR.mkdir(parents=True, exist_ok=True)

RWF_DIR = DATASETS_ROOT / 'RWF-2000'
RLVS_DIR = DATASETS_ROOT / 'RLVS' / 'Real Life Violence Dataset'
TRAIN_ROOT = RWF_DIR
TEST_ROOT = RLVS_DIR

CHECKPOINT_DIR = PHASE3_MODEL_DIR / 'checkpoints_rwf_only_on_rlvs'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PREFIX = 'rwf_only'
EPOCH_CHECKPOINT_PATTERN = str(CHECKPOINT_DIR / (CHECKPOINT_PREFIX + '_epoch_{epoch:03d}.weights.h5'))
TRAINING_LOG = str(PHASE3_MODEL_DIR / 'training_log_rwf_only.csv')
LOCAL_MODEL_OUTPUT = str(PHASE3_MODEL_DIR / 'sirch_model_rwf_only.weights.h5')
EVAL_CSV = str(PHASE3_MODEL_DIR / 'evaluation_rwf_only_on_rlvs.csv')
EVAL_DETAILS_CSV = str(PHASE3_MODEL_DIR / 'evaluation_rwf_only_on_rlvs_details.csv')

random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

print('Parametres SIRCH P3-G charges.')
print(f'Experiment : {EXPERIMENT_NAME}')
print(f'Train domain : {TRAIN_DOMAIN} -> {TRAIN_ROOT}')
print(f'Test domain : {TEST_DOMAIN} -> {TEST_ROOT}')
print(f'Checkpoints : {CHECKPOINT_DIR}')
print(f'Modele final : {LOCAL_MODEL_OUTPUT}')
print(f'Log CSV : {TRAINING_LOG}')


In [ ]:
# ============================================================
# CELLULE 2 - Indexer les videos et garantir zero leakage
# ============================================================
VIDEO_EXTENSIONS = ('*.mp4', '*.avi', '*.mov', '*.mkv')


def collect_from_folder(folder, label):
    files = []
    for ext in VIDEO_EXTENSIONS:
        files.extend(glob.glob(os.path.join(folder, '**', ext), recursive=True))
    return [(str(Path(path).resolve()), label) for path in files]


def collect_dataset(root):
    root = Path(root)
    if not root.exists():
        raise FileNotFoundError(f'Dataset introuvable : {root}')
    samples = []
    label_rules = {
        1: ['Violence', 'Fight'],
        0: ['NonViolence', 'NonFight', 'Non-Violence', 'Non Violence'],
    }
    for label, names in label_rules.items():
        for name in names:
            for folder in glob.glob(os.path.join(str(root), '**', name), recursive=True):
                if os.path.isdir(folder):
                    samples.extend(collect_from_folder(folder, label))
    unique = {}
    for path, label in samples:
        unique[path] = label
    result = list(unique.items())
    if not result:
        raise RuntimeError(f'Aucune video trouvee dans {root}')
    return result

train_samples = collect_dataset(TRAIN_ROOT)
target_test_samples = collect_dataset(TEST_ROOT)
random.shuffle(train_samples)
random.shuffle(target_test_samples)

train_source_paths = [path for path, _ in train_samples]
train_source_labels = [label for _, label in train_samples]

test_paths = [path for path, _ in target_test_samples]
test_labels = [label for _, label in target_test_samples]

train_paths, val_paths, train_labels, val_labels = train_test_split(
    train_source_paths,
    train_source_labels,
    test_size=VALIDATION_SIZE,
    random_state=42,
    stratify=train_source_labels,
)

train_val_set = set(map(str.lower, train_paths + val_paths))
test_set = set(map(str.lower, test_paths))
overlap = train_val_set.intersection(test_set)
if overlap:
    raise RuntimeError(f'Data leakage detecte entre train/val et test : {len(overlap)} fichiers en commun')

print(f'Train source total ({TRAIN_DOMAIN}) : {len(train_source_paths)} clips')
print(f'Train : {len(train_paths)} clips | Violence={sum(train_labels)} | Non-violence={len(train_labels) - sum(train_labels)}')
print(f'Validation : {len(val_paths)} clips | Violence={sum(val_labels)} | Non-violence={len(val_labels) - sum(val_labels)}')
print(f'Test cible ({TEST_DOMAIN}) : {len(test_paths)} clips | Violence={sum(test_labels)} | Non-violence={len(test_labels) - sum(test_labels)}')
print('Verification leakage : aucun fichier commun entre train/val et test cible.')


In [ ]:
# ============================================================
# CELLULE 3 - Generateur video memoire-efficace
# ============================================================
class VideoSequence(tf.keras.utils.Sequence):
    def __init__(self, video_paths, labels, batch_size=BATCH_SIZE, shuffle=True):
        self.video_paths = list(video_paths)
        self.labels = np.array(labels, dtype=np.float32)
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.indices = np.arange(len(self.video_paths))
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.video_paths) / self.batch_size))

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)

    def __getitem__(self, idx):
        batch_indices = self.indices[idx * self.batch_size:(idx + 1) * self.batch_size]
        batch_x = np.zeros((len(batch_indices), N_FRAMES, IMG_SIZE, IMG_SIZE, 3), dtype=np.float32)
        batch_y = self.labels[batch_indices]
        for i, sample_idx in enumerate(batch_indices):
            batch_x[i] = load_video_frames(self.video_paths[sample_idx])
        return batch_x, batch_y


def load_video_frames(video_path):
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        cap.release()
        return np.zeros((N_FRAMES, IMG_SIZE, IMG_SIZE, 3), dtype=np.float32)

    frame_indices = np.linspace(0, max(total - 1, 0), N_FRAMES).astype(int)
    frames = []
    for frame_index in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_index))
        ok, frame = cap.read()
        if not ok:
            frame = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
        else:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame = cv2.resize(frame, (IMG_SIZE, IMG_SIZE))
        frames.append(frame.astype(np.float32))
    cap.release()
    frames = np.stack(frames, axis=0)
    return tf.keras.applications.efficientnet.preprocess_input(frames)

train_gen = VideoSequence(train_paths, train_labels, shuffle=True)
val_gen = VideoSequence(val_paths, val_labels, shuffle=False)
test_gen = VideoSequence(test_paths, test_labels, shuffle=False)
print('Generateurs prets.')


In [ ]:
# ============================================================
# CELLULE 4 - Architecture EfficientNetB0 + LSTM identique au modele SIRCH
# ============================================================
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import EfficientNetB0

base_model = EfficientNetB0(
    weights='imagenet',
    include_top=False,
    pooling='avg',
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
)
base_model.trainable = False

sequence_input = layers.Input(shape=(N_FRAMES, IMG_SIZE, IMG_SIZE, 3))
features = layers.TimeDistributed(base_model)(sequence_input)
x = layers.LSTM(LSTM_UNITS, return_sequences=False)(features)
x = layers.Dropout(DROPOUT)(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.3)(x)
output = layers.Dense(1, activation='sigmoid')(x)

model = Model(inputs=sequence_input, outputs=output)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss=LOSS,
    metrics=['accuracy', tf.keras.metrics.Precision(name='precision'), tf.keras.metrics.Recall(name='recall')],
)
model.summary()


In [ ]:
# ============================================================
# CELLULE 5 - Entrainement avec reprise automatique et callbacks de securite
# ============================================================
def checkpoint_epoch(path):
    match = re.search(rf'{re.escape(CHECKPOINT_PREFIX)}_epoch_(\d+)\.weights\.h5$', str(path))
    return int(match.group(1)) if match else -1

checkpoint_files = sorted(
    set(glob.glob(str(CHECKPOINT_DIR / f'{CHECKPOINT_PREFIX}_epoch_*.weights.h5'))),
    key=checkpoint_epoch,
)
initial_epoch = 0
if checkpoint_files:
    latest_checkpoint = checkpoint_files[-1]
    initial_epoch = checkpoint_epoch(latest_checkpoint)
    print(f'Reprise depuis les poids du checkpoint : {latest_checkpoint}')
    model.load_weights(latest_checkpoint)
else:
    print('Aucun checkpoint trouve. Entrainement depuis le debut.')

best_val_loss = None
if Path(TRAINING_LOG).exists() and Path(LOCAL_MODEL_OUTPUT).exists():
    with open(TRAINING_LOG, newline='') as file:
        for row in csv.DictReader(file):
            value = row.get('val_loss')
            if value not in (None, ''):
                value = float(value)
                best_val_loss = value if best_val_loss is None else min(best_val_loss, value)

epoch_checkpoint = tf.keras.callbacks.ModelCheckpoint(
    EPOCH_CHECKPOINT_PATTERN,
    save_best_only=False,
    save_weights_only=True,
    verbose=1,
)

best_model_kwargs = dict(
    filepath=LOCAL_MODEL_OUTPUT,
    monitor='val_loss',
    mode='min',
    save_best_only=True,
    save_weights_only=True,
    verbose=1,
)
if best_val_loss is not None:
    best_model_kwargs['initial_value_threshold'] = best_val_loss
best_model_checkpoint = tf.keras.callbacks.ModelCheckpoint(**best_model_kwargs)

csv_logger = tf.keras.callbacks.CSVLogger(TRAINING_LOG, append=True)
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=6,
    restore_best_weights=True,
    verbose=1,
)
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1,
)

print('Verification callbacks :')
print('- ModelCheckpoint epoch : save_best_only=False, save_weights_only=True')
print('- ModelCheckpoint best : monitor=val_loss, save_best_only=True')
print('- EarlyStopping : monitor=val_loss, patience=6, restore_best_weights=True')
print('- CSVLogger :', TRAINING_LOG)
print('- Reprise automatique : initial_epoch =', initial_epoch)

if initial_epoch >= EPOCHS:
    print(f'Entrainement deja arrive a {initial_epoch} epochs sur {EPOCHS}.')
else:
    history = model.fit(
        train_gen,
        validation_data=val_gen,
        initial_epoch=initial_epoch,
        epochs=EPOCHS,
        callbacks=[epoch_checkpoint, best_model_checkpoint, csv_logger, early_stop, reduce_lr],
    )

if not Path(LOCAL_MODEL_OUTPUT).exists():
    model.save_weights(LOCAL_MODEL_OUTPUT)
print(f'Meilleurs poids sauvegardes : {LOCAL_MODEL_OUTPUT}')


In [ ]:
# ============================================================
# CELLULE 6 - Evaluation cross-dataset sur le dataset cible complet
# ============================================================
if Path(LOCAL_MODEL_OUTPUT).exists():
    model.load_weights(LOCAL_MODEL_OUTPUT)
    print(f'Poids du meilleur modele charges : {LOCAL_MODEL_OUTPUT}')
else:
    print('Aucun meilleur modele sauvegarde trouve. Evaluation avec les poids courants en memoire.')

start = time.time()
scores = model.predict(test_gen, verbose=1).ravel()
elapsed = time.time() - start
preds = (scores >= THRESHOLD).astype(int)
truth = np.array(test_labels, dtype=int)

accuracy = accuracy_score(truth, preds)
precision = precision_score(truth, preds, zero_division=0)
recall = recall_score(truth, preds, zero_division=0)
f1 = f1_score(truth, preds, zero_division=0)
auc = roc_auc_score(truth, scores) if len(np.unique(truth)) == 2 else float('nan')
cm = confusion_matrix(truth, preds, labels=[0, 1])
ms_per_frame = (elapsed / max(len(test_paths) * N_FRAMES, 1)) * 1000

print(f'Experiment : {EXPERIMENT_NAME}')
print(f'Train : {TRAIN_DOMAIN} uniquement')
print(f'Test : {TEST_DOMAIN} uniquement')
print(f'Accuracy : {accuracy:.4f}')
print(f'Precision : {precision:.4f}')
print(f'Recall : {recall:.4f}')
print(f'F1-score : {f1:.4f}')
print(f'ROC-AUC : {auc:.4f}')
print(f'Temps inference moyen : {ms_per_frame:.2f} ms/frame')
print('Matrice de confusion [TN FP; FN TP] :')
print(cm)

with open(EVAL_CSV, 'w', newline='', encoding='utf-8') as file:
    writer = csv.writer(file)
    writer.writerow(['experiment', 'train_domain', 'test_domain', 'threshold', 'n_test', 'tn', 'fp', 'fn', 'tp', 'accuracy', 'precision', 'recall', 'f1_score', 'roc_auc', 'ms_per_frame', 'model_path', 'training_log'])
    writer.writerow([
        EXPERIMENT_NAME, TRAIN_DOMAIN, TEST_DOMAIN, THRESHOLD, len(test_paths),
        int(cm[0, 0]), int(cm[0, 1]), int(cm[1, 0]), int(cm[1, 1]),
        f'{accuracy:.6f}', f'{precision:.6f}', f'{recall:.6f}', f'{f1:.6f}', f'{auc:.6f}', f'{ms_per_frame:.6f}',
        LOCAL_MODEL_OUTPUT, TRAINING_LOG,
    ])

with open(EVAL_DETAILS_CSV, 'w', newline='', encoding='utf-8') as file:
    writer = csv.writer(file)
    writer.writerow(['experiment', 'video', 'true_label', 'score', 'pred_label'])
    for path, y, score, pred in zip(test_paths, truth, scores, preds):
        writer.writerow([EXPERIMENT_NAME, path, int(y), f'{float(score):.8f}', int(pred)])

print(f'Resultats sauvegardes : {EVAL_CSV}')
print(f'Details video par video : {EVAL_DETAILS_CSV}')
